# Extract packet-capture CSVs into encoder-ready instance CSVs

Converts the three son-tung Kaggle datasets (one packet per row, `;` separated)
into the format read by the SupCon / binary-MLP pipelines:

- one CSV row = one instance = one source file;
- `MAX_PACKETS = 10,000` packets x `[label_id, relative_time, direction, packet_size]` = 40,000 comma-separated numbers, no header;
- unused slots are padded with `0,0,0,0` (padding mask is `packet_size == 0`);
- label = name of the file's parent folder (ids start at 1, id 0 is reserved like in the reference `label_mapping.csv`); train/test split is done per **file** so no capture appears in both splits.

The three datasets share one schema (`protocol;length;relative_time;direction;src_ip;src_port;dst_ip;dst_port`), so a single extractor handles all of them.
Output per dataset: `train_data.csv`, `test_data.csv`, `label_mapping.csv`, `manifest.csv`, `report.json` under `/kaggle/working/extracted/<name>/`.

## 1. Configuration

In [ ]:
# OVERVIEW: dataset list, encoder contract and the two semantic switches (time, direction).
import csv
import json
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("BKCS_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("BKCS_OUTPUT_ROOT", "/kaggle/working/extracted"))

# "kaggle_slug" / "title" are the URL slug and the display name shown in the notebook's Input panel;
# a mounted folder matches either one (case and -/_ differences are ignored).
# "role" is only recorded in report.json / dataset_roles.json (known | unknown | unspecified);
# it does not change the extraction. Set "path" to override automatic lookup.
DATASETS = [
    {"name": "warp_masque", "kaggle_slug": "warp-masque-datasaet", "title": "WARP_masque_datasaet", "role": "unspecified", "path": None},
    {"name": "swizerland", "kaggle_slug": "swizerland-dataset-new", "title": "Swizerland_dataset_new", "role": "unspecified", "path": None},
    {"name": "masque_500", "kaggle_slug": "500-masque-new", "title": "500_masque_new", "role": "unspecified", "path": None},
]

# Encoder contract (must match RawPacketEncoder / pretrain_AOL.pth).
MAX_PACKETS = 10_000
FIELDS_PER_PACKET = 4          # [label_id, relative_time, direction, packet_size]

# Source schema (header of every input CSV).
SOURCE_DELIMITER = ";"
REQUIRED_COLUMNS = ("length", "relative_time", "direction")

# --- Semantic switches: verify against a row of the AOL training CSV before training. ---
# "cumulative_from_first": relative_time = seconds since the first packet (first = 0, never decreasing).
#                           The project docs describe the AOL feature this way.
# "as_is":                  copy the source column unchanged (source values are inter-packet gaps).
TIME_MODE = "cumulative_from_first"
# Maps source direction -> value written to the instance CSV. Source: 1 = client->server, 0 = server->client.
DIRECTION_MAP = {0: 0, 1: 1}   # e.g. {0: -1, 1: 1} if the AOL data uses +/-1

# Label ids start at 1: id 0 is reserved for the unnamed/padding entry that the reference
# label_mapping.csv carries as its first row (",0").
LABEL_ID_START = 1

TEST_RATIO = 0.2               # fraction of FILES per label sent to test
SEED = 42
FILE_PATTERN = "*.csv"
MIN_FILES_PER_LABEL = 1        # labels with fewer files are dropped (reported)
# Random (seeded) cap on files per label BEFORE the train/test split; None keeps every file.
# 500 labels x 300 files = 150k files per dataset would not fit /kaggle/working (~20 GB) nor finish in time.
MAX_FILES_PER_LABEL = 100
DRY_RUN = False                # True: scan + report only, write nothing

# Run ONE dataset per notebook session (the 3 together do not fit /kaggle/working, ~20 GB):
# set RUN_ONLY to a dataset "name" from DATASETS, e.g. "warp_masque"; None runs all of them.
RUN_ONLY = None
# Stop writing (cleanly, keeping what is written) once the output reaches this size; /kaggle/working holds ~20 GB.
# Rows are written interleaved across labels, so stopping early leaves a uniform fraction of every label.
MAX_OUTPUT_GB = 19.0

assert TIME_MODE in ("cumulative_from_first", "as_is")
if RUN_ONLY is not None:
    selected = {RUN_ONLY} if isinstance(RUN_ONLY, str) else set(RUN_ONLY)
    unknown = selected - {spec["name"] for spec in DATASETS}
    assert not unknown, f"RUN_ONLY has unknown dataset name(s) {sorted(unknown)}"
    DATASETS = [spec for spec in DATASETS if spec["name"] in selected]
print("Datasets this run:", [spec["name"] for spec in DATASETS])
assert 0.0 <= TEST_RATIO < 1.0

## 2. Locate datasets and index files

In [ ]:
# OVERVIEW: find each attached Kaggle dataset directory and list its CSV files with their label (parent folder).
import fnmatch
import time


def _norm(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", name.lower())


def _dirs_up_to(root: Path, max_depth: int):
    """Yield (depth, directory) below root, breadth-first, without descending into data folders."""
    level = [root]
    for depth in range(1, max_depth + 1):
        nxt = []
        for parent in level:
            for child in sorted(parent.iterdir()):
                if child.is_dir():
                    yield depth, child
                    nxt.append(child)
        level = nxt


def locate_dataset(spec: dict) -> Path:
    if spec.get("path"):
        path = Path(spec["path"])
        if not path.is_dir():
            raise FileNotFoundError(f"{spec['name']}: configured path does not exist: {path}")
        return path
    wanted = {_norm(spec["kaggle_slug"]), _norm(spec.get("title", ""))} - {""}
    found = [(depth, p) for depth, p in _dirs_up_to(INPUT_ROOT, 3) if _norm(p.name) in wanted]
    if found:
        shallowest = min(depth for depth, _ in found)
        matches = [p for depth, p in found if depth == shallowest]
    else:
        matches = []
    if len(matches) != 1:
        seen = [str(p.relative_to(INPUT_ROOT)) for _, p in _dirs_up_to(INPUT_ROOT, 2)] if INPUT_ROOT.is_dir() else []
        raise FileNotFoundError(
            f"{spec['name']}: expected one directory named like {spec['kaggle_slug']!r} or {spec.get('title')!r} "
            f"under {INPUT_ROOT}; found {matches}. Set 'path' explicitly. Folders seen: {seen}"
        )
    return matches[0]


def index_files(root: Path, dataset_name: str) -> dict[str, list[Path]]:
    """label (immediate parent folder; dataset name for files in the root) -> sorted files."""
    # os.walk reads file types from the directory listing, so there is no per-file stat() call
    # (Path.rglob + is_file() issued one stat per file, which is very slow on /kaggle/input).
    by_label: dict[str, list[Path]] = defaultdict(list)
    started, folders = time.time(), 0
    for dirpath, _, filenames in os.walk(root):
        folders += 1
        names = sorted(n for n in filenames if fnmatch.fnmatch(n, FILE_PATTERN) and n not in ("label_mapping.csv", "manifest.csv"))
        if names:
            label = dataset_name if Path(dirpath) == root else Path(dirpath).name
            by_label[label].extend(Path(dirpath, n) for n in names)
        if folders % 50 == 0:
            print(f"    ...scanned {folders} folders, {sum(map(len, by_label.values()))} files, {time.time() - started:.0f}s", flush=True)
    return dict(by_label)


for spec in DATASETS:
    root = locate_dataset(spec)
    print(f"{spec['name']:<12} {root}", flush=True)
    files = index_files(root, spec["name"])
    available = sum(map(len, files.values()))
    if MAX_FILES_PER_LABEL is not None:
        rng = random.Random(f"{SEED}:{spec['name']}")
        files = {
            label: sorted(rng.sample(paths, MAX_FILES_PER_LABEL)) if len(paths) > MAX_FILES_PER_LABEL else paths
            for label, paths in files.items()
        }
    spec["root"] = root
    spec["files"] = files
    spec["files_available"] = available
    print(f"  labels={len(files)} files={sum(map(len, files.values()))} (available {available})", flush=True)
    for label, paths in list(files.items())[:8]:
        print(f"    {label!r}: {len(paths)} file(s), e.g. {paths[0].name}")

# One output row is at most ~0.4 MB (10,000 packets x 4 numbers); Kaggle /kaggle/working holds ~20 GB.
total_files = sum(len(p) for spec in DATASETS for p in spec["files"].values())
print(f"\nTOTAL files={total_files}  output upper bound~{total_files * 0.4 / 1e3:.1f} GB")
if total_files * 0.4 / 1e3 > MAX_OUTPUT_GB:
    print(f"NOTE: worst-case output exceeds {MAX_OUTPUT_GB} GB; the extraction stops cleanly at MAX_OUTPUT_GB, keeping an even fraction of every label (see section 5).")

## 3. Parse one capture into packet rows

In [ ]:
# OVERVIEW: read one source CSV, validate it and convert it to (time, direction, size) packet tuples.
def parse_capture(path: Path) -> tuple[list[tuple[float, float, float]], int]:
    """Return (packets, number of packets whose time had to be clamped)."""
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, delimiter=SOURCE_DELIMITER)
        header = next(reader, None)
        if header is None:
            raise ValueError(f"{path}: empty file")
        header = [h.strip() for h in header]
        missing = [c for c in REQUIRED_COLUMNS if c not in header]
        if missing:
            raise ValueError(f"{path}: missing columns {missing}; header={header}")
        i_len, i_time, i_dir = (header.index(c) for c in REQUIRED_COLUMNS)
        rows: list[tuple[float, float, float]] = []
        for line_no, row in enumerate(reader, start=2):
            if not row:
                continue
            try:
                length = float(row[i_len])
                gap = float(row[i_time])
                direction = int(float(row[i_dir]))
            except (ValueError, IndexError) as error:
                raise ValueError(f"{path}:{line_no}: bad row {row!r} ({error})") from error
            if direction not in DIRECTION_MAP:
                raise ValueError(f"{path}:{line_no}: direction {direction} not in DIRECTION_MAP")
            if length < 0:
                raise ValueError(f"{path}:{line_no}: negative length in {row!r}")
            rows.append((gap, float(DIRECTION_MAP[direction]), length))
    if not rows:
        raise ValueError(f"{path}: no packets")
    # Source gaps can be slightly negative when the capture reorders packets (e.g. -0.000014).
    # The encoder expects non-negative time, so clamp and report how often it happened.
    clamped = 0
    if TIME_MODE == "cumulative_from_first":
        total, previous, first = 0.0, 0.0, rows[0][0]
        converted = []
        for gap, direction, length in rows:
            total += gap
            current = total - first
            if current < previous:
                current, clamped = previous, clamped + 1
            previous = current
            converted.append((current, direction, length))
        rows = converted
    else:
        clamped = sum(1 for gap, _, _ in rows if gap < 0)
        rows = [(max(gap, 0.0), direction, length) for gap, direction, length in rows]
    return rows, clamped


def fmt(value: float) -> str:
    """Compact text for one number (integral values stay integers)."""
    if value == int(value):
        return str(int(value))
    return f"{value:.9g}"


PAD = ",0,0,0,0"


def encode_row(label_id: int, packets: list[tuple[float, float, float]], clamped: int = 0) -> tuple[str, dict]:
    kept = packets[:MAX_PACKETS]
    parts = [f"{label_id},{fmt(t)},{fmt(d)},{fmt(s)}" for t, d, s in kept]
    text = ",".join(parts) + PAD * (MAX_PACKETS - len(kept))
    zero_size = sum(1 for _, _, s in kept if s == 0)
    return text, {"packets": len(packets), "kept": len(kept), "truncated": len(packets) > MAX_PACKETS,
                  "zero_size_packets": zero_size, "clamped_time_packets": clamped}

## 4. File-level stratified split

In [ ]:
# OVERVIEW: split FILES (not packets) per label so a capture never lands in both train and test.
def split_files(files: dict[str, list[Path]], seed: int, test_ratio: float):
    rng = random.Random(seed)
    train, test, dropped = [], [], {}
    for label in sorted(files):
        paths = list(files[label])
        if len(paths) < MIN_FILES_PER_LABEL:
            dropped[label] = len(paths)
            continue
        rng.shuffle(paths)
        n_test = int(round(len(paths) * test_ratio)) if len(paths) >= 2 else 0
        n_test = min(n_test, len(paths) - 1)   # keep at least one training file
        test.extend((label, p) for p in paths[:n_test])
        train.extend((label, p) for p in paths[n_test:])
    return train, test, dropped

## 5. Write instance CSVs, mapping and manifest

In [ ]:
# OVERVIEW: write rows in an interleaved order and stop cleanly once the output budget is used up.
BUDGET = {"bytes": 0}     # bytes written by this run, shared by all datasets (one /kaggle/working disk)


def interleave(train, test, seed):
    """Order (split, label, path) so that every label advances at the same pace.

    Cutting the list at any point leaves the same fraction of every label, with the train/test ratio intact.
    """
    rng = random.Random(f"{seed}:order")
    by_label = defaultdict(list)
    for split_name, items in (("train", train), ("test", test)):
        for label, path in items:
            by_label[label].append((split_name, path))
    keyed = []
    for label, entries in by_label.items():
        rng.shuffle(entries)
        keyed.extend(((i + rng.random()) / len(entries), split_name, label, path)
                     for i, (split_name, path) in enumerate(entries))
    keyed.sort(key=lambda item: item[0])
    return [(split_name, label, path) for _, split_name, label, path in keyed]


# OVERVIEW: extract one dataset to train_data.csv / test_data.csv / label_mapping.csv / manifest.csv / report.json.
def extract_dataset(spec: dict) -> dict:
    name, files = spec["name"], spec["files"]
    train, test, dropped = split_files(files, SEED, TEST_RATIO)
    labels = sorted({label for label, _ in train + test})
    label_ids = {label: index for index, label in enumerate(labels, start=LABEL_ID_START)}
    out_dir = OUTPUT_ROOT / name
    report = {
        "name": name, "role": spec["role"], "source_root": str(spec["root"]),
        "time_mode": TIME_MODE, "direction_map": {str(k): v for k, v in DIRECTION_MAP.items()},
        "max_packets": MAX_PACKETS, "seed": SEED, "test_ratio": TEST_RATIO,
        "labels": len(labels), "dropped_labels": dropped,
        "planned_train_instances": len(train), "planned_test_instances": len(test),
        "train_instances": 0, "test_instances": 0, "stopped_by_size": False,
        "truncated_instances": 0, "zero_size_packets": 0, "clamped_time_packets": 0,
    }
    if DRY_RUN:
        print(f"[dry-run] {name}: labels={len(labels)} train={len(train)} test={len(test)}")
        report["train_instances"], report["test_instances"] = len(train), len(test)
        return report
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    packet_counts = []
    line_numbers = {"train": 0, "test": 0}
    order = interleave(train, test, SEED)
    limit_bytes = MAX_OUTPUT_GB * 1e9
    started = time.time()
    handles = {split: (out_dir / f"{split}_data.csv").open("w", encoding="ascii", newline="") for split in line_numbers}
    try:
        for done, (split_name, label, path) in enumerate(order, start=1):
            packets, clamped = parse_capture(path)
            text, info = encode_row(label_ids[label], packets, clamped)
            if BUDGET["bytes"] + len(text) + 1 > limit_bytes:
                report["stopped_by_size"] = True
                print(f"    {name}: STOP at {BUDGET['bytes'] / 1e9:.2f} GB (limit {MAX_OUTPUT_GB} GB) "
                      f"after {done - 1}/{len(order)} files; remaining files are not written", flush=True)
                break
            handles[split_name].write(text + "\n")
            BUDGET["bytes"] += len(text) + 1
            line_numbers[split_name] += 1
            report["truncated_instances"] += int(info["truncated"])
            report["zero_size_packets"] += info["zero_size_packets"]
            report["clamped_time_packets"] += info["clamped_time_packets"]
            packet_counts.append(info["packets"])
            manifest_rows.append({
                "split": split_name, "line_number": line_numbers[split_name], "label_id": label_ids[label],
                "label_name": label, "source_file": str(path), "packets": info["packets"],
                "kept": info["kept"], "truncated": info["truncated"],
                "clamped_time_packets": info["clamped_time_packets"],
            })
            if done % 2000 == 0:
                print(f"    {name}: {done}/{len(order)} files, {BUDGET['bytes'] / 1e9:.2f} GB written, "
                      f"{time.time() - started:.0f}s", flush=True)
    finally:
        for handle in handles.values():
            handle.close()
    report["train_instances"], report["test_instances"] = line_numbers["train"], line_numbers["test"]
    report["output_gb"] = round(BUDGET["bytes"] / 1e9, 3)
    with (out_dir / "label_mapping.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["label_name", "label_id"])      # same column order as the reference mapping
        if LABEL_ID_START == 1:
            writer.writerow(["", 0])                         # reserved id 0 (empty name), as in the reference
        writer.writerows((label, label_ids[label]) for label in labels)
    with (out_dir / "manifest.csv").open("w", encoding="utf-8", newline="") as handle:
        fieldnames = ["split", "line_number", "label_id", "label_name", "source_file", "packets", "kept", "truncated", "clamped_time_packets"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(manifest_rows)
    report["packets_min"] = min(packet_counts) if packet_counts else 0
    report["packets_mean"] = sum(packet_counts) / len(packet_counts) if packet_counts else 0.0
    report["packets_max"] = max(packet_counts) if packet_counts else 0
    report["instances_per_label"] = dict(Counter(row["label_name"] for row in manifest_rows))
    (out_dir / "report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    return report

## 6. Verify the written files against the encoder contract

In [ ]:
# OVERVIEW: re-read every output row and check width, label, padding layout and non-decreasing time.
def verify_output(name: str) -> dict:
    out_dir = OUTPUT_ROOT / name
    mapping = {}
    with (out_dir / "label_mapping.csv").open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            mapping[int(row["label_id"])] = row["label_name"]
    mapping.pop(0, None)    # reserved id, never a real label
    rows = 0
    for split_name in ("train", "test"):
        path = out_dir / f"{split_name}_data.csv"
        with path.open("r", encoding="ascii") as handle:
            for line_no, line in enumerate(handle, start=1):
                values = line.rstrip("\n").split(",")
                if len(values) != MAX_PACKETS * FIELDS_PER_PACKET:
                    raise AssertionError(f"{path}:{line_no}: {len(values)} values")
                packets = [values[i:i + 4] for i in range(0, len(values), 4)]
                n_real = next((i for i, p in enumerate(packets) if p == ["0", "0", "0", "0"]), len(packets))
                real, tail = packets[:n_real], packets[n_real:]
                if not real:
                    raise AssertionError(f"{path}:{line_no}: instance has no packets")
                if any(p != ["0", "0", "0", "0"] for p in tail):
                    raise AssertionError(f"{path}:{line_no}: real packet found after padding")
                if any(float(p[3]) == 0 for p in real):
                    raise AssertionError(f"{path}:{line_no}: real packet with packet_size 0")
                label = int(float(real[0][0]))
                if label not in mapping or any(int(float(p[0])) != label for p in real):
                    raise AssertionError(f"{path}:{line_no}: label mismatch")
                if TIME_MODE == "cumulative_from_first":
                    times = [float(p[1]) for p in real]
                    if times[0] != 0 or any(b < a for a, b in zip(times, times[1:])):
                        raise AssertionError(f"{path}:{line_no}: time is not a non-decreasing series from 0")
                rows += 1
    return {"rows": rows}

## 7. Run

In [ ]:
# OVERVIEW: extract all datasets, verify them and write dataset_roles.json.
reports = {}
for spec in DATASETS:
    r = reports[spec["name"]] = extract_dataset(spec)
    print(f"{spec['name']}: labels={r['labels']} train={r['train_instances']} test={r['test_instances']} "
          f"truncated={r['truncated_instances']} zero_size_packets={r['zero_size_packets']} "
          f"clamped_time_packets={r['clamped_time_packets']}")
    if r["stopped_by_size"]:
        kept = r["train_instances"] + r["test_instances"]
        planned = r["planned_train_instances"] + r["planned_test_instances"]
        print(f"  stopped by size limit: wrote {kept}/{planned} files ({kept / planned:.0%}), {r['output_gb']} GB")
    if r["dropped_labels"]:
        print("  dropped labels:", r["dropped_labels"])
    if r["test_instances"] == 0 and not DRY_RUN:
        print("  WARNING: no test instances (labels with a single file, or the size limit was reached first); add files, lower MIN_FILES_PER_LABEL or raise MAX_OUTPUT_GB")
    if not DRY_RUN:
        print("  verify:", verify_output(spec["name"]))
if not DRY_RUN:
    (OUTPUT_ROOT / "dataset_roles.json").write_text(
        json.dumps({s["name"]: s["role"] for s in DATASETS}, indent=2), encoding="utf-8")
    print("Saved:", OUTPUT_ROOT)